# Array-CDF Playground

Interactive exploration of empirical-CDF figures (`sinr_cdf`, `scnr_cdf`)
from **SLURM job-array** runs that have been aggregated by
`scripts/aggregate_array_batch.py`.

**Why a separate notebook?** Array runs produce a per-experiment tree like:

```
results/exp_sinr_cdf/
├── array_52671898_task_0/        per-task outputs
├── array_52671898_task_1/
├── ...
├── array_52671898_aggregated/    ← merged result (this is what we load)
└── latest -> array_52671898_task_9/  (after rsync, often points at the wrong dir!)
```

The `latest` symlink is unreliable after `sync_results_from_hpc3.sh` — sync
order can land it on a per-task dir.  This notebook **never** uses `latest`
for array runs.  It picks the most recent `array_<jobid>_aggregated/`
explicitly (highest job ID), or you can pin a specific `ARRAY_ID`.

**Compatible experiments:** `sinr_cdf`, `scnr_cdf`.

In [ ]:
# Shared setup: make `cordis` importable from notebooks/ + IEEE rcParams.
import sys, logging
from pathlib import Path
from _playground_helpers import (
    setup_paper_style,
    load_aggregated_array_result, array_summary,
    print_per_task_summary, per_task_summary,
)
import numpy as np
import matplotlib.pyplot as plt

# Set use_latex=False if pdflatex isn't on PATH on this machine.
setup_paper_style(use_latex=True)
logging.basicConfig(level=logging.WARNING, format='%(levelname)-7s %(message)s')

## 1. Load aggregated array result

Set `exp_name` to the family (`'sinr'` or `'scnr'`) and `ARRAY_ID` if you
want to pin a specific SLURM job ID.  Leave `ARRAY_ID = None` to grab the
most recent array (highest job ID under `results/exp_<name>_cdf/`).

In [ ]:
# ── The two knobs ──
exp_name  = 'sinr'              # 'sinr' or 'scnr'
ARRAY_ID  = None                # e.g. '52671898'; None = most recent

# Derived names — change `exp_name` above and these update automatically.
EXPERIMENT   = f'{exp_name}_cdf'
METRIC_MIN   = f'min_{exp_name}_db'
METRIC_MEAN  = f'mean_{exp_name}_db'
METRIC_DISPLAY = 'SINR' if exp_name == 'sinr' else 'SCNR'

result, array_id = load_aggregated_array_result(EXPERIMENT, array_id=ARRAY_ID)
print(f'\nResolved ARRAY_ID = {array_id}')

## 2. Inspect aggregated metadata + per-task breakdown

`array_summary` prints the total trial count, drops, realizations,
and the per-task table (task_id, seed, trials, drops, realizations).

Seeds in `_array_common.sh` follow `SEED = BASE_SEED + ARRAY_TASK_ID`,
so they should form a contiguous range — e.g. `42-51` for a 10-task run
with the default `BASE_SEED=42`.  A gap signals a task that failed and
wasn't re-submitted.

In [ ]:
array_summary(result, EXPERIMENT, array_id)

Programmatic access to the same data via `per_task_summary` (returns
a list of dicts):

In [ ]:
rows = per_task_summary(EXPERIMENT, array_id)
print(f'{len(rows)} task records.')
if rows:
    print(f'First task: {rows[0]}')
# Total-trials cross-check.
total = sum(r['n_trials'] for r in rows)
print(f'Sum of per-task trials: {total}')

## 3. Quick CDF — all algorithms

The aggregated `result.sim_result` has the merged per-trial arrays
(`min_sinr_per_trial`, etc.) — these concatenate cleanly across
tasks, so the CDF spans the full `n_trials_total`.

In [ ]:
from cordis.plotting import plot_cdf, figsize

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_cdf(result.sim_result, metric=METRIC_MIN, ax=ax)
ax.set_title(f'{EXPERIMENT} (array {array_id}): min-{METRIC_DISPLAY} CDF')
ax.set_xlabel(f'min-{METRIC_DISPLAY} [dB]')
plt.show()

## 4. Filter algorithms via `only=`

Restrict the plot to a hand-picked algorithm subset — same syntax as the
non-array CDF playground.

In [ ]:
PICK = ['CORDIS-Split', 'CORDIS-ADMM', 'Centralized']

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_cdf(result.sim_result, metric=METRIC_MIN, ax=ax, only=PICK)
ax.set_title(f'{EXPERIMENT} (array {array_id}): CORDIS vs. Centralized')
ax.set_xlabel(f'min-{METRIC_DISPLAY} [dB]')
plt.show()

## 5. SINR floor (γ) + infeasibility annotation

`plot_cdf` (Stage 20) supports three γ-related kwargs:

- **`gamma_db=<value>`**: draws a vertical dashed line at γ and, with the
  default `annotate_infeasibility=True`, appends `(inf=X.X%)` to each
  algorithm's legend label showing the fraction of trials with
  `min_sinr_db < γ`.
- **`show_feasible_only=True`**: replaces the full CDF with a
  conditional CDF over only the trials where the algorithm satisfied
  the floor (`min_sinr_db >= γ`).  Requires `gamma_db` to be set;
  warns and falls back to the full CDF if every trial is infeasible.
- **`annotate_infeasibility=False`**: turns off the legend annotation
  if you want clean labels.

The annotation correctly handles `text.usetex=True` (auto-escapes `%`
so it isn't eaten as a LaTeX comment).

In [ ]:
# Full CDFs annotated with infeasibility rate at γ = 5 dB.
GAMMA_DB = 5.0

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_cdf(
    result.sim_result, metric=METRIC_MIN, ax=ax, only=PICK,
    gamma_db=GAMMA_DB,
    show_feasible_only=False,        # full CDF
    annotate_infeasibility=True,     # show (inf=X.X%) in legend
)
ax.set_title(f'{EXPERIMENT}: full CDF with γ={GAMMA_DB:.0f} dB')
ax.set_xlabel(f'min-{METRIC_DISPLAY} [dB]')
plt.show()

In [ ]:
# Feasible-only conditional CDF: only trials where each algorithm
# satisfied the γ floor.  Useful for showing CORDIS-ADMM's behavior
# on the trials where it actually solved the problem, separated from
# its infeasibility tail.
fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_cdf(
    result.sim_result, metric=METRIC_MIN, ax=ax, only=PICK,
    gamma_db=GAMMA_DB,
    show_feasible_only=True,         # ← conditional only
    annotate_infeasibility=True,     # legend still shows infeasibility rate
)
ax.set_title(f'{EXPERIMENT}: feasible-only CDF at γ={GAMMA_DB:.0f} dB')
ax.set_xlabel(f'min-{METRIC_DISPLAY} [dB]')
plt.show()

## 6. Min vs. mean comparison

Pair the worst-user metric (`min_{family}_db`) with the average-user
metric (`mean_{family}_db`) in a 2-up layout.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=figsize(width='double', aspect=3/2))
plot_cdf(result.sim_result, metric=METRIC_MIN, ax=axes[0], only=PICK)
axes[0].set_title(f'Worst-user {METRIC_DISPLAY}')
axes[0].set_xlabel(f'min-{METRIC_DISPLAY} [dB]')
plot_cdf(result.sim_result, metric=METRIC_MEAN, ax=axes[1], only=PICK)
axes[1].set_title(f'Average-user {METRIC_DISPLAY}')
axes[1].set_xlabel(f'mean-{METRIC_DISPLAY} [dB]')
plt.tight_layout()
plt.show()

## 7. Save with provenance metadata

In [ ]:
from cordis.plotting import save_figure

out = save_figure(
    fig,
    base_path=f'../figures/playground/{EXPERIMENT}_array_{array_id}',
    formats=('pdf', 'png'),
    metadata={
        'Experiment': EXPERIMENT,
        'ArrayID':    str(array_id),
        'Notebook':   'playground_array_cdf',
    },
)
for p in out:
    print('wrote', p)

## 8. Compare multiple array runs (optional)

Set `ARRAY_A` / `ARRAY_B` to two array IDs to overlay their curves
(e.g. before vs. after a configuration change).  Leave as `None` to skip.

In [ ]:
ARRAY_A = None    # e.g. '52671898'
ARRAY_B = None    # e.g. '52680001'

if ARRAY_A and ARRAY_B:
    res_a, _ = load_aggregated_array_result(EXPERIMENT, array_id=ARRAY_A)
    res_b, _ = load_aggregated_array_result(EXPERIMENT, array_id=ARRAY_B)
    fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
    n_before = len(ax.lines)
    plot_cdf(res_a.sim_result, metric=METRIC_MIN, ax=ax, only=PICK)
    n_after_a = len(ax.lines)
    for ln in ax.lines[n_before:n_after_a]:
        ln.set_label(f'{ln.get_label()} (A={ARRAY_A})')
    plot_cdf(res_b.sim_result, metric=METRIC_MIN, ax=ax, only=PICK)
    for ln in ax.lines[n_after_a:]:
        ln.set_linestyle('--')
        ln.set_label(f'{ln.get_label()} (B={ARRAY_B})')
    ax.legend()
    ax.set_title(f'{EXPERIMENT}: array {ARRAY_A} vs. {ARRAY_B}')
    ax.set_xlabel(f'min-{METRIC_DISPLAY} [dB]')
    plt.show()
else:
    print('Set ARRAY_A and ARRAY_B above to enable multi-array overlay.')